# Notebook 22 — Option contract and PnL engine (Stage 1 only)

This notebook freezes the common economic contract and accounting environment for Application 1. It maps the frozen N21 TEST episodes to **availability-only** post-signal opportunities, builds a transparent Garman–Kohlhagen spot-ATM O/N straddle engine, and tests explicit hourly delta-hedge cashflows.

The economic object is a **model-based delta-hedged USD/JPY ATM O/N straddle PnL implied by observed Bloomberg ATM volatility**. It is not a historical executable dealer option PnL. Stage 2 detection, forecast bridges, trading rules, policy selection, ML, and TEST option economics are deliberately out of scope.

In [1]:
from pathlib import Path
from datetime import time, datetime, timezone
from zoneinfo import ZoneInfo
import json
import math

import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display, Markdown

ROOT_CANDIDATES = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "CODEWORK" / "Linus' Task Re-Do"]
ROOT = next(p for p in ROOT_CANDIDATES if (p / "Data" / "interim").exists())
DATA = ROOT / "Data"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"
FIGURES = ROOT / "figures"
PROCESSED.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

NY_TZ = "America/New_York"
NY = ZoneInfo(NY_TZ)
ASSUMED_IV_TIME = time(10, 0)
ASSUMED_EXPIRY_TIME = time(10, 0)
ATM_METHOD = "SPOT_ATM"
RATE_METHOD = "ZERO_DOMESTIC_AND_FOREIGN"
DAY_COUNT_METHOD = "ACT_365"
HEDGE_FREQUENCY = "HOURLY_OBSERVED_CLOSES"
HEDGE_VOL_METHOD = "ENTRY_VOL_FIXED"
SPOT_COST_METHOD = "OBSERVED_BID_ASK"
OPTION_COST_METHOD = "NONE_AVAILABLE"
ALLOW_STALE_IV = False
ALLOW_SPOT_INTERPOLATION = False
ALLOW_IV_INTERPOLATION = False
TEST_ECONOMICS_UNLOCKED = False

ATM_TICKER = "USDJPYVON BGN Curncy"
ATM_FIELD = "PX_LAST"
ATM_TENOR = "O/N / 1D ATM"
ACCOUNTING_UNIT = "JPY per USD underlying notional"
RECON_TOL = 1e-10
RUN_ID = datetime.now(timezone.utc).isoformat()

assert TEST_ECONOMICS_UNLOCKED is False
assert not ALLOW_STALE_IV and not ALLOW_SPOT_INTERPOLATION and not ALLOW_IV_INTERPOLATION
print(f"Project root: {ROOT}")
print("Stage 1 configuration loaded; TEST economics are LOCKED.")

Project root: C:\Users\Rajiv Nawal\OneDrive\Documents\GITREPOS\HSBC-AN-RN\CODEWORK\Linus' Task Re-Do
Stage 1 configuration loaded; TEST economics are LOCKED.


## 1. Stage 1 data inventory and provenance

The audited Bloomberg series is the project’s USD/JPY O/N / 1D ATM volatility quote. Its raw `iv` field is a percentage/vol-point annualised volatility quote; only the pricing engine converts it to decimal volatility as `sigma0 = raw_atm_vol / 100`. The weekday-normalised `iv_model_var` is not an option-pricing input.

The audited hourly spot timestamp denotes the **start** of a one-hour candle. Therefore its close becomes observable one hour later, implemented explicitly as `candle_end_time_utc = timestamp + 1 hour`. Direct raw bid and ask closes are joined to the audited midpoint on the exact candle-start timestamp; no quote interpolation or repair is performed.

In [2]:
ATM_AUDITED_PATH = ROOT / "archive_notebook14_results" / "usdjpy_atm_tidy.csv"
ATM_RAW_PATH = DATA / "ATM_Volatility_-_all_currencies_diff_tenors.xlsx"
HOURLY_AUDITED_PATH = INTERIM / "usdjpy_hourly_mid_audited.csv"
HOURLY_BID_PATH = DATA / "usdjpy_hourly_data" / "usdjpy_hourly_bid.csv"
HOURLY_ASK_PATH = DATA / "usdjpy_hourly_data" / "usdjpy_hourly_ask.csv"
N21_EPISODES_PATH = PROCESSED / "21_hidden_event_episodes.csv"
N21_DAILY_PATH = PROCESSED / "21_hidden_event_daily_scores.csv"
CANONICAL_PANEL_PATH = PROCESSED / "16_empirical_analysis_panel.csv"

required_paths = [
    ATM_AUDITED_PATH, ATM_RAW_PATH, HOURLY_AUDITED_PATH, HOURLY_BID_PATH,
    HOURLY_ASK_PATH, N21_EPISODES_PATH, N21_DAILY_PATH, CANONICAL_PANEL_PATH
]
assert all(p.exists() for p in required_paths), [str(p) for p in required_paths if not p.exists()]

iv_all = pd.read_csv(ATM_AUDITED_PATH)
required_atm_columns = {"date", "atm_iv", "currency_pair", "tenor", "ticker"}
assert required_atm_columns.issubset(iv_all.columns), "Clean audited ATM extract has an unexpected schema."
iv_all = iv_all.loc[
    iv_all["currency_pair"].eq("USDJPY")
    & iv_all["tenor"].eq("1D")
    & iv_all["ticker"].eq(ATM_TICKER)
].copy()
iv_all["date"] = pd.to_datetime(iv_all["date"], errors="coerce").dt.normalize()
iv_all["iv"] = pd.to_numeric(iv_all["atm_iv"], errors="coerce")
iv_valid = iv_all.loc[iv_all["date"].notna() & iv_all["iv"].gt(0), ["date", "iv"]].copy()
assert not iv_valid.empty
assert iv_valid["date"].duplicated().sum() == 0
assert iv_valid["date"].min() == pd.Timestamp("1998-12-14")
assert iv_valid["date"].max() == pd.Timestamp("2026-07-01")
assert not iv_valid["date"].dt.year.eq(1970).any()
assert iv_valid["iv"].gt(0).all()

hourly_mid = pd.read_csv(HOURLY_AUDITED_PATH)
hourly_mid["timestamp_parsed"] = pd.to_datetime(hourly_mid["timestamp_parsed"], utc=True)
hourly_mid = hourly_mid.sort_values("timestamp_parsed").reset_index(drop=True)
for c in ["mid_open", "mid_high", "mid_low", "mid_close", "spread_close"]:
    hourly_mid[c] = pd.to_numeric(hourly_mid[c], errors="coerce")

bid = pd.read_csv(HOURLY_BID_PATH, usecols=["timestamp", "close"])
ask = pd.read_csv(HOURLY_ASK_PATH, usecols=["timestamp", "close"])
bid["timestamp_parsed"] = pd.to_datetime(bid.pop("timestamp"), utc=True)
ask["timestamp_parsed"] = pd.to_datetime(ask.pop("timestamp"), utc=True)
bid = bid.rename(columns={"close": "bid_close"})
ask = ask.rename(columns={"close": "ask_close"})
bid["bid_close"] = pd.to_numeric(bid["bid_close"], errors="coerce")
ask["ask_close"] = pd.to_numeric(ask["ask_close"], errors="coerce")

hourly = (hourly_mid.merge(bid, on="timestamp_parsed", how="left", validate="one_to_one")
          .merge(ask, on="timestamp_parsed", how="left", validate="one_to_one"))
assert len(hourly) == len(hourly_mid)
hourly["candle_end_time_utc"] = hourly["timestamp_parsed"] + pd.Timedelta(hours=1)
hourly["candle_end_time_ny"] = hourly["candle_end_time_utc"].dt.tz_convert(NY_TZ)
hourly["candle_end_date_ny"] = hourly["candle_end_time_ny"].dt.date
hourly["quote_valid"] = (
    hourly[["mid_close", "bid_close", "ask_close"]].notna().all(axis=1)
    & hourly["mid_close"].gt(0)
    & hourly["bid_close"].gt(0)
    & hourly["ask_close"].gt(0)
    & hourly["ask_close"].ge(hourly["bid_close"])
)
quote_overlap = hourly[["mid_close", "spread_close", "bid_close", "ask_close"]].dropna()
midpoint_error = (quote_overlap["mid_close"] - (quote_overlap["bid_close"] + quote_overlap["ask_close"]) / 2).abs()
spread_error = (quote_overlap["spread_close"] - (quote_overlap["ask_close"] - quote_overlap["bid_close"])).abs()
assert midpoint_error.max() < 1e-10 and spread_error.max() < 1e-10

def exact_quote_rows_at(clock):
    """Return exact candle-end observations at a New York clock time and their valid-quote subset."""
    exact = hourly.loc[
        hourly["candle_end_time_ny"].dt.hour.eq(clock.hour)
        & hourly["candle_end_time_ny"].dt.minute.eq(clock.minute)
        & hourly["candle_end_time_ny"].dt.second.eq(clock.second)
    ].copy()
    assert exact["candle_end_date_ny"].duplicated().sum() == 0
    assert (exact["candle_end_time_utc"] - exact["timestamp_parsed"]).eq(pd.Timedelta(hours=1)).all()
    valid = exact.loc[exact["quote_valid"].astype(bool)].copy()
    assert valid["quote_valid"].all()
    assert (valid["ask_close"] >= valid["bid_close"]).all()
    assert valid["candle_end_date_ny"].duplicated().sum() == 0
    return exact, valid

entry_quotes, entry_quotes_valid = exact_quote_rows_at(ASSUMED_IV_TIME)
expiry_quotes, expiry_quotes_valid = exact_quote_rows_at(ASSUMED_EXPIRY_TIME)

known_bad_quote_time = pd.Timestamp("2008-03-02 21:00:00+00:00")
if known_bad_quote_time in set(hourly["timestamp_parsed"]):
    assert not hourly.loc[hourly["timestamp_parsed"].eq(known_bad_quote_time), "quote_valid"].iloc[0]
    assert known_bad_quote_time + pd.Timedelta(hours=1) not in set(entry_quotes_valid["candle_end_time_utc"])
    assert known_bad_quote_time + pd.Timedelta(hours=1) not in set(expiry_quotes_valid["candle_end_time_utc"])

canonical = pd.read_csv(CANONICAL_PANEL_PATH)
canonical["model_day"] = pd.to_datetime(canonical["model_day"]).dt.normalize()
canonical_days = set(canonical["model_day"].dropna().dt.date)
valid_iv_dates = set(iv_valid["date"].dt.date)
all_entry_quote_dates = set(entry_quotes["candle_end_date_ny"])
valid_entry_quote_dates = set(entry_quotes_valid["candle_end_date_ny"])
all_expiry_quote_dates = set(expiry_quotes["candle_end_date_ny"])
valid_expiry_quote_dates = set(expiry_quotes_valid["candle_end_date_ny"])

inventory = pd.DataFrame([
    {
        "component": "ATM implied volatility",
        "source_file": str(ATM_AUDITED_PATH.relative_to(ROOT)),
        "raw_provenance_file": str(ATM_RAW_PATH.relative_to(ROOT)),
        "ticker_or_timestamp_fields": f"{ATM_TICKER}; {ATM_FIELD}; audited source column=atm_iv (renamed iv internally)",
        "tenor_or_timezone": ATM_TENOR,
        "units_or_candle_semantics": "annualised volatility points; intraday timestamp absent",
        "first_valid_observation": str(iv_valid["date"].min().date()),
        "last_valid_observation": str(iv_valid["date"].max().date()),
        "n_observations": len(iv_valid),
        "duplicate_keys": int(iv_valid["date"].duplicated().sum()),
        "missing_vs_canonical_chronology": len(canonical_days - valid_iv_dates),
        "gap_audit_fields": "clean audited tidy extract; source sheet/ticker retained",
        "intraday_timestamp_observed": False,
    },
    {
        "component": "USD/JPY hourly spot",
        "source_file": str(HOURLY_AUDITED_PATH.relative_to(ROOT)),
        "raw_provenance_file": f"{HOURLY_BID_PATH.relative_to(ROOT)}; {HOURLY_ASK_PATH.relative_to(ROOT)}",
        "ticker_or_timestamp_fields": "timestamp; timestamp_parsed; exact timestamp join for bid_close/ask_close",
        "tenor_or_timezone": "UTC source; converted DST-aware to America/New_York",
        "units_or_candle_semantics": "timestamp is candle start; OHLC midpoint; close observed at start + 1 hour",
        "first_valid_observation": str(hourly["timestamp_parsed"].min()),
        "last_valid_observation": str(hourly["timestamp_parsed"].max()),
        "n_observations": len(hourly),
        "duplicate_keys": int(hourly["timestamp_parsed"].duplicated().sum()),
        "missing_vs_canonical_chronology": len(canonical_days - valid_entry_quote_dates),
        "gap_audit_fields": "is_adjacent_hour_diagnostic; gap_classification; major_coverage_segment",
        "intraday_timestamp_observed": True,
    },
])
display(inventory)
print({
    "exact_hour_spacing_rows": int(hourly["timestamp_parsed"].diff().eq(pd.Timedelta(hours=1)).sum()),
    "crossed_or_invalid_quote_rows": int((~hourly["quote_valid"]).sum()),
    "entry_quote_dates_before_validity_filter": len(all_entry_quote_dates),
    "entry_quote_dates_after_validity_filter": len(valid_entry_quote_dates),
    "expiry_quote_dates_before_validity_filter": len(all_expiry_quote_dates),
    "expiry_quote_dates_after_validity_filter": len(valid_expiry_quote_dates),
})

,component,source_file,raw_provenance_file,ticker_or_timestamp_fields,tenor_or_timezone,units_or_candle_semantics,first_valid_observation,last_valid_observation,n_observations,duplicate_keys,missing_vs_canonical_chronology,gap_audit_fields,intraday_timestamp_observed
0,ATM implied volatility,archive_notebook14_results\usdjpy_atm_tidy.csv,Data\ATM_Volatility_-_all_currencies_diff_teno...,USDJPYVON BGN Curncy; PX_LAST; audited source ...,O/N / 1D ATM,annualised volatility points; intraday timesta...,1998-12-14,2026-07-01,6452,0,325,clean audited tidy extract; source sheet/ticke...,False
1,USD/JPY hourly spot,Data\interim\usdjpy_hourly_mid_audited.csv,Data\usdjpy_hourly_data\usdjpy_hourly_bid.csv;...,timestamp; timestamp_parsed; exact timestamp j...,UTC source; converted DST-aware to America/New...,timestamp is candle start; OHLC midpoint; clos...,2003-05-04 20:00:00+00:00,2026-07-01 23:00:00+00:00,149842,0,217,is_adjacent_hour_diagnostic; gap_classificatio...,True


{'exact_hour_spacing_rows': 148735, 'crossed_or_invalid_quote_rows': 1, 'entry_quote_dates_before_validity_filter': 6192, 'entry_quote_dates_after_validity_filter': 6192, 'expiry_quote_dates_before_validity_filter': 6192, 'expiry_quote_dates_after_validity_filter': 6192}


## 2. Observed data versus modelling assumptions

> **Two separate 10:00 assumptions**
>
> 1. **IV observation / valuation time:** a Bloomberg daily O/N ATM observation dated (E) is treated as available at assumed (E) 10:00 New York. The historical extract contains no intraday timestamp.
> 2. **Synthetic expiry cut:** the model option expires at the next later date with an observed exact 10:00 New York spot close.
>
> Neither time is documented by the Bloomberg extract, and they are not presented as one Bloomberg convention.

In [3]:
observed_vs_assumed = pd.DataFrame([
    ("N21 hidden-state type", "Frozen empirical output"),
    ("Signal availability at D 17:00 NY", "Determined by frozen model-day construction"),
    ("Raw Bloomberg O/N ATM IV", "Observed"),
    ("IV quote date", "Observed"),
    ("IV intraday timestamp", "Not observed"),
    ("10:00 NY IV availability time", "Assumption"),
    ("Hourly spot", "Observed"),
    ("Hourly spot spread / bid-ask", "Observed; exact timestamp-matched bid/ask closes"),
    ("Entry spot", "Observed under assumed entry timestamp"),
    ("Exact Bloomberg O/N expiry convention", "Not observed"),
    ("Next-tradable 10:00 expiry", "Assumption"),
    ("ATM strike K=S0", "Spot-ATM synthetic assumption"),
    ("Rates", "Assumed zero domestic and foreign"),
    ("Option premium", "Garman–Kohlhagen model-implied"),
    ("Option bid/ask", "Not observed"),
    ("Hourly hedge rule", "Strategy assumption: observed closes only"),
    ("Spot hedge costs", "Observed bid/ask execution"),
], columns=["component", "status"])
display(observed_vs_assumed)

,component,status
0,N21 hidden-state type,Frozen empirical output
1,Signal availability at D 17:00 NY,Determined by frozen model-day construction
2,Raw Bloomberg O/N ATM IV,Observed
3,IV quote date,Observed
4,IV intraday timestamp,Not observed
5,10:00 NY IV availability time,Assumption
6,Hourly spot,Observed
7,Hourly spot spread / bid-ask,Observed; exact timestamp-matched bid/ask closes
8,Entry spot,Observed under assumed entry timestamp
9,Exact Bloomberg O/N expiry convention,Not observed


## 3. Frozen N21 signal chronology and TEST-safe feasibility

A frozen hidden episode is first knowable at 17:00 New York on its **first candidate day**, not at its retrospectively selected peak. Entry is the first later calendar date having both a same-date raw O/N ATM observation and an exact observed candle close at 10:00 New York. Expiry is the first still-later exact observed 10:00 close. This section uses only dates, timestamps, availability flags, and audited gap/quote-validity flags; it never accesses TEST spot or IV levels.

In [4]:
episodes = pd.read_csv(N21_EPISODES_PATH)
daily_scores = pd.read_csv(N21_DAILY_PATH)
episodes["start_day"] = pd.to_datetime(episodes["start_day"]).dt.normalize()
episodes["end_day"] = pd.to_datetime(episodes["end_day"]).dt.normalize()
daily_scores["realised_model_day"] = pd.to_datetime(daily_scores["realised_model_day"]).dt.normalize()
candidate_daily = daily_scores.loc[daily_scores["candidate_type"].isin(["REALISED_ONLY", "IMPLIED_ONLY"])].copy()

assert len(candidate_daily) == 18
assert len(episodes) == 17
assert episodes["candidate_type"].value_counts().to_dict() == {"IMPLIED_ONLY": 13, "REALISED_ONLY": 4}

# Independent reconstruction from the frozen daily candidates without changing membership.
reconstructed_starts = []
for ep in episodes.itertuples():
    members = candidate_daily.loc[
        candidate_daily["candidate_type"].eq(ep.candidate_type)
        & candidate_daily["realised_model_day"].between(ep.start_day, ep.end_day),
        "realised_model_day"
    ]
    assert len(members) == int(ep.n_days)
    reconstructed_starts.append(members.min())
assert pd.Series(reconstructed_starts).reset_index(drop=True).equals(episodes["start_day"].reset_index(drop=True))

eligible_entry_dates = sorted(valid_iv_dates & valid_entry_quote_dates)
expiry_observation_dates = sorted(valid_expiry_quote_dates)
time_audit = hourly[[
    "candle_end_time_utc", "candle_end_time_ny", "gap_classification", "quote_valid"
]].copy()

def local_time_on(day, clock):
    return pd.Timestamp(day).tz_localize(NY_TZ) + pd.Timedelta(hours=clock.hour, minutes=clock.minute)

def first_after(sorted_dates, day):
    return next((d for d in sorted_dates if d > day), None)

safe_rows = []
for ep in episodes.itertuples():
    signal_day = ep.start_day.date()
    signal_ny = local_time_on(signal_day, time(17, 0))
    entry_date = first_after(eligible_entry_dates, signal_day)
    entry_ny = local_time_on(entry_date, ASSUMED_IV_TIME) if entry_date else pd.NaT
    entry_utc = entry_ny.tz_convert("UTC") if entry_date else pd.NaT
    expiry_date = first_after(expiry_observation_dates, entry_date) if entry_date else None
    expiry_ny = local_time_on(expiry_date, ASSUMED_EXPIRY_TIME) if expiry_date else pd.NaT
    expiry_utc = expiry_ny.tz_convert("UTC") if expiry_date else pd.NaT

    path_audit = time_audit.iloc[0:0]
    if entry_date and expiry_date:
        path_audit = time_audit.loc[
            time_audit["candle_end_time_utc"].between(entry_utc, expiry_utc)
        ].copy()
    transitions = path_audit.iloc[1:]
    unexpected = int(transitions["gap_classification"].isin([
        "short unexpected weekday/other gap", "major historical discontinuity", "other interval"
    ]).sum())
    expected_weekend = bool(transitions["gap_classification"].eq(
        "expected Friday-to-Sunday/Monday closure"
    ).any())
    path_available = bool(
        entry_date and expiry_date and len(path_audit) >= 2
        and unexpected == 0 and path_audit["quote_valid"].all()
    )
    evaluable = bool(entry_date and expiry_date and path_available)
    if not entry_date:
        reason = "No later date has both same-date raw O/N ATM IV and exact observed 10:00 NY spot close."
    elif not expiry_date:
        reason = "No later exact observed 10:00 NY spot close exists for expiry."
    elif unexpected:
        reason = "Observed hedge path contains an unexpected non-weekend source gap."
    elif not path_audit["quote_valid"].all():
        reason = "Observed hedge path contains an invalid/crossed spot quote."
    else:
        reason = ""

    elapsed_hours = (expiry_utc - entry_utc).total_seconds() / 3600 if evaluable else np.nan
    safe_rows.append({
        "episode_id": ep.episode_id,
        "candidate_type": ep.candidate_type,
        "signal_model_day": signal_day,
        "signal_time_ny": signal_ny,
        "entry_date": entry_date,
        "assumed_iv_observation_time_ny": entry_ny,
        "option_entry_time_ny": entry_ny,
        "option_entry_time_utc": entry_utc,
        "expiry_date": expiry_date,
        "expiry_time_ny": expiry_ny,
        "expiry_time_utc": expiry_utc,
        "elapsed_calendar_hours": elapsed_hours,
        "elapsed_calendar_days": elapsed_hours / 24 if evaluable else np.nan,
        "entry_iv_available": bool(entry_date and entry_date in valid_iv_dates),
        "entry_spot_available": bool(entry_date and entry_date in valid_entry_quote_dates),
        "expiry_spot_available": bool(expiry_date and expiry_date in valid_expiry_quote_dates),
        "n_observed_hedge_points": len(path_audit) if entry_date and expiry_date else 0,
        "unexpected_gap_count": unexpected,
        "has_expected_weekend_gap": expected_weekend,
        "hedge_path_available": path_available,
        "evaluable": evaluable,
        "unevaluable_reason": reason,
    })

test_feasibility = pd.DataFrame(safe_rows)
BANNED_TEST_FRAGMENTS = [
    "premium", "strike", "sigma", "delta", "payoff", "pnl", "transaction_cost",
    "spread_cost", "turnover", "entry_spot_level", "terminal_spot", "raw_iv_level"
]
assert not any(fragment in c.lower() for c in test_feasibility.columns for fragment in BANNED_TEST_FRAGMENTS)
assert len(test_feasibility) == 17 and int(test_feasibility["evaluable"].sum()) == 16
assert test_feasibility.loc[test_feasibility["episode_id"].eq("E21_017"), "evaluable"].eq(False).all()
assert test_feasibility.loc[test_feasibility["episode_id"].eq("E21_017"), "unevaluable_reason"].str.len().gt(0).all()
assert test_feasibility.loc[test_feasibility["evaluable"], "option_entry_time_ny"].gt(
    test_feasibility.loc[test_feasibility["evaluable"], "signal_time_ny"]
).all()
assert all(pd.Timestamp(e).date() > pd.Timestamp(d).date() for e, d in zip(
    test_feasibility.loc[test_feasibility["evaluable"], "entry_date"],
    test_feasibility.loc[test_feasibility["evaluable"], "signal_model_day"]
))
display(test_feasibility[["episode_id", "candidate_type", "signal_model_day", "entry_date",
                          "expiry_date", "elapsed_calendar_hours", "unexpected_gap_count",
                          "has_expected_weekend_gap", "evaluable", "unevaluable_reason"]])

,episode_id,candidate_type,signal_model_day,entry_date,expiry_date,elapsed_calendar_hours,unexpected_gap_count,has_expected_weekend_gap,evaluable,unevaluable_reason
0,E21_001,REALISED_ONLY,2021-11-26,2021-11-29,2021-11-30,24.0,0,False,True,
1,E21_002,IMPLIED_ONLY,2022-03-02,2022-03-03,2022-03-04,24.0,0,False,True,
2,E21_003,IMPLIED_ONLY,2022-10-17,2022-10-18,2022-10-19,24.0,0,False,True,
3,E21_004,IMPLIED_ONLY,2023-09-12,2023-09-13,2023-09-14,24.0,0,False,True,
4,E21_005,IMPLIED_ONLY,2023-10-04,2023-10-05,2023-10-06,24.0,0,False,True,
5,E21_006,REALISED_ONLY,2023-12-07,2023-12-08,2023-12-11,72.0,0,True,True,
6,E21_007,IMPLIED_ONLY,2024-04-11,2024-04-12,2024-04-15,72.0,0,True,True,
7,E21_008,IMPLIED_ONLY,2024-04-16,2024-04-17,2024-04-18,24.0,0,False,True,
8,E21_009,IMPLIED_ONLY,2024-04-25,2024-04-26,2024-04-29,72.0,0,True,True,
9,E21_010,IMPLIED_ONLY,2024-04-30,2024-05-01,2024-05-02,24.0,0,False,True,


## 4. Pricing contract and TEST embargo

The synthetic contract uses (K=S_0), zero JPY domestic and USD foreign rates, raw entry O/N ATM volatility divided by 100, and ACT/365 actual assumed calendar time. Fixed entry volatility is retained throughout hedging. All cash amounts are JPY per unit USD underlying-equivalent option notional.

Every real-data pricing or ledger entry point calls the TEST guard. Synthetic unit tests may bypass it explicitly.

In [5]:
EMBARGO_MESSAGE = "TEST option economics are sealed until Stages 2 and 3 are frozen."

def assert_economics_allowed(sample_split, synthetic=False):
    if synthetic:
        return
    if str(sample_split).lower() == "test" and not TEST_ECONOMICS_UNLOCKED:
        raise RuntimeError(EMBARGO_MESSAGE)

def garman_kohlhagen(S, K, tau, sigma, r_domestic=0.0, r_foreign=0.0,
                     *, sample_split, synthetic=False):
    assert_economics_allowed(sample_split, synthetic=synthetic)
    values = {"S": S, "K": K, "tau": tau, "sigma": sigma,
              "r_domestic": r_domestic, "r_foreign": r_foreign}
    if not all(np.isfinite(float(v)) for v in values.values()):
        raise ValueError("All pricing inputs must be finite.")
    if S <= 0:
        raise ValueError("S must be strictly positive.")
    if K <= 0:
        raise ValueError("K must be strictly positive.")
    if tau <= 0:
        raise ValueError("tau must be strictly positive.")
    if sigma <= 0:
        raise ValueError("sigma must be strictly positive.")

    root_tau = math.sqrt(tau)
    d1 = (math.log(S / K) + (r_domestic - r_foreign + 0.5 * sigma**2) * tau) / (sigma * root_tau)
    d2 = d1 - sigma * root_tau
    df_d = math.exp(-r_domestic * tau)
    df_f = math.exp(-r_foreign * tau)
    call = S * df_f * norm.cdf(d1) - K * df_d * norm.cdf(d2)
    put = K * df_d * norm.cdf(-d2) - S * df_f * norm.cdf(-d1)
    call_delta = df_f * norm.cdf(d1)
    put_delta = df_f * (norm.cdf(d1) - 1.0)
    return {
        "call_value": float(call),
        "put_value": float(put),
        "straddle_value": float(call + put),
        "call_delta": float(call_delta),
        "put_delta": float(put_delta),
        "straddle_delta": float(call_delta + put_delta),
    }

def _execution_price(q, bid_px, ask_px, mid_px, *, sample_split, synthetic=False):
    assert_economics_allowed(sample_split, synthetic=synthetic)
    if not all(np.isfinite([bid_px, ask_px, mid_px])):
        raise ValueError("Execution quotes must be finite.")
    if min(bid_px, ask_px, mid_px) <= 0:
        raise ValueError("Execution quotes must be strictly positive.")
    if ask_px < bid_px:
        raise ValueError("ask must be greater than or equal to bid; negative spread is invalid.")
    return ask_px if q > 0 else bid_px if q < 0 else mid_px

def run_delta_hedge(path, *, K, sigma0, position, sample_split,
                    r_domestic=0.0, r_foreign=0.0, synthetic=False):
    assert_economics_allowed(sample_split, synthetic=synthetic)
    if position not in {"long", "short"}:
        raise ValueError("position must be 'long' or 'short'.")
    required = {"time", "mid", "bid", "ask"}
    if not required.issubset(path.columns):
        raise ValueError(f"path must contain {sorted(required)}.")
    p = path[list(required)].copy().sort_values("time").reset_index(drop=True)
    p["time"] = pd.to_datetime(p["time"], utc=True)
    if len(p) < 2 or p["time"].duplicated().any() or not p["time"].is_monotonic_increasing:
        raise ValueError("path must contain at least two unique increasing timestamps.")
    for c in ["mid", "bid", "ask"]:
        p[c] = pd.to_numeric(p[c], errors="coerce")
    if not np.isfinite(p[["mid", "bid", "ask"]].to_numpy()).all():
        raise ValueError("path quotes must be finite.")
    if (p[["mid", "bid", "ask"]] <= 0).any().any():
        raise ValueError("path quotes must be strictly positive.")
    if (p["ask"] < p["bid"]).any():
        raise ValueError("ask must be greater than or equal to bid; negative spread is invalid.")
    if not np.allclose(p["mid"], (p["bid"] + p["ask"]) / 2, atol=1e-10, rtol=0):
        raise ValueError("mid must equal the bid/ask average.")

    entry_time, expiry_time = p["time"].iloc[0], p["time"].iloc[-1]
    tau0 = (expiry_time - entry_time).total_seconds() / (365.0 * 24.0 * 3600.0)
    if tau0 <= 0:
        raise ValueError("Expiry must be strictly after entry.")
    entry_pricing = garman_kohlhagen(
        p["mid"].iloc[0], K, tau0, sigma0, r_domestic, r_foreign,
        sample_split=sample_split, synthetic=synthetic
    )
    option_sign = 1.0 if position == "long" else -1.0
    holding = 0.0
    ledger_rows = []

    for j, row in p.iloc[:-1].iterrows():
        remaining_tau = (expiry_time - row["time"]).total_seconds() / (365.0 * 24.0 * 3600.0)
        pricing = garman_kohlhagen(
            row["mid"], K, remaining_tau, sigma0, r_domestic, r_foreign,
            sample_split=sample_split, synthetic=synthetic
        )
        target = -option_sign * pricing["straddle_delta"]
        q = target - holding
        execution = _execution_price(q, row["bid"], row["ask"], row["mid"],
                                     sample_split=sample_split, synthetic=synthetic)
        holding = target
        ledger_rows.append({
            "time": row["time"],
            "transaction_type": "initial_hedge" if j == 0 else "intermediate_rebalance",
            "remaining_tau": remaining_tau,
            "sigma_hedge": sigma0,
            "delta_target": pricing["straddle_delta"],
            "desired_hedge_holding": target,
            "trade_q": q,
            "mid": row["mid"], "bid": row["bid"], "ask": row["ask"],
            "execution_price": execution,
            "holding_after": holding,
        })

    terminal = p.iloc[-1]
    q_terminal = -holding
    terminal_execution = _execution_price(
        q_terminal, terminal["bid"], terminal["ask"], terminal["mid"],
        sample_split=sample_split, synthetic=synthetic
    )
    holding += q_terminal
    ledger_rows.append({
        "time": terminal["time"],
        "transaction_type": "terminal_unwind",
        "remaining_tau": 0.0,
        "sigma_hedge": sigma0,
        "delta_target": np.nan,
        "desired_hedge_holding": np.nan,
        "trade_q": q_terminal,
        "mid": terminal["mid"], "bid": terminal["bid"], "ask": terminal["ask"],
        "execution_price": terminal_execution,
        "holding_after": holding,
    })
    ledger = pd.DataFrame(ledger_rows)
    ledger["spot_cashflow_mid"] = -ledger["trade_q"] * ledger["mid"]
    ledger["spot_cashflow_bidask"] = -ledger["trade_q"] * ledger["execution_price"]
    ledger["half_spread_cost"] = ledger["trade_q"].abs() * (ledger["ask"] - ledger["bid"]) / 2.0

    premium = entry_pricing["straddle_value"]
    payoff = abs(float(terminal["mid"]) - K)
    option_entry_cashflow = -option_sign * premium
    option_terminal_cashflow = option_sign * payoff
    pnl_mid = option_entry_cashflow + ledger["spot_cashflow_mid"].sum() + option_terminal_cashflow
    pnl_bidask = option_entry_cashflow + ledger["spot_cashflow_bidask"].sum() + option_terminal_cashflow
    spread_cost = ledger["half_spread_cost"].sum()
    summary = {
        "position": position,
        "entry_time": entry_time,
        "expiry_time": expiry_time,
        "maturity_hours": (expiry_time - entry_time).total_seconds() / 3600.0,
        "premium": premium,
        "terminal_payoff": payoff,
        "spot_hedge_cashflow_mid": ledger["spot_cashflow_mid"].sum(),
        "spot_hedge_spread_cost": spread_cost,
        "pnl_mid": pnl_mid,
        "pnl_bidask": pnl_bidask,
        "premium_normalised_pnl_mid": pnl_mid / premium,
        "premium_normalised_pnl_bidask": pnl_bidask / premium,
        "final_hedge_inventory": holding,
        "n_ledger_transactions": len(ledger),
    }
    if abs(pnl_bidask - (pnl_mid - spread_cost)) >= RECON_TOL:
        raise AssertionError("Explicit bid/ask PnL does not reconcile to midpoint PnL less half-spread cost.")
    if abs(holding) >= RECON_TOL:
        raise AssertionError("Terminal unwind did not close the spot hedge inventory.")
    return summary, ledger

In [6]:
# Both protected economic entry points use a frozen TEST episode identity and deliberately fabricated non-market inputs.
_embargo_episode_id = test_feasibility.iloc[0]["episode_id"]
pricing_embargo_guard_passed = False
try:
    garman_kohlhagen(
        1.0, 1.0, 1 / 365, 0.10,
        sample_split="test", synthetic=False
    )
except RuntimeError as exc:
    assert str(exc) == EMBARGO_MESSAGE
    pricing_embargo_guard_passed = True
assert pricing_embargo_guard_passed
print("PASS — TEST pricing guard blocked economics.")

hedge_embargo_guard_passed = False
embargo_probe_path = pd.DataFrame({
    "time": pd.to_datetime(["2025-01-01 15:00Z", "2025-01-02 15:00Z"], utc=True),
    "mid": [1.0, 1.0],
    "bid": [0.99, 0.99],
    "ask": [1.01, 1.01],
})
try:
    run_delta_hedge(
        embargo_probe_path, K=1.0, sigma0=0.10, position="long",
        sample_split="test", synthetic=False
    )
except RuntimeError as exc:
    assert str(exc) == EMBARGO_MESSAGE
    hedge_embargo_guard_passed = True
assert hedge_embargo_guard_passed
print("PASS — TEST hedge/PnL guard blocked economics.")

PASS — TEST pricing guard blocked economics.
PASS — TEST hedge/PnL guard blocked economics.


## 5. Explicit hedge-accounting contract

For a long straddle, the desired USD holding is minus the model straddle delta; for a short straddle it is plus that delta. The ledger records initial establishment, each observed pre-expiry rebalance, and one terminal unwind. It does **not** create a fresh expiry delta target.

Midpoint accounting is option entry cashflow plus (-sum q_j S_j) plus terminal straddle payoff. Bid/ask accounting executes USD purchases at ask and sales at bid. The independent half-spread sum includes every hedge transaction and must reconcile exactly to the difference between midpoint and after-spot-cost PnL.

## 6. Synthetic option-engine unit tests

In [7]:
unit_results = []

def record_test(name, fn):
    fn()
    unit_results.append({"test": name, "status": "PASS"})

def synthetic_path(times, mids, spread=0.0):
    times = pd.to_datetime(times, utc=True)
    mids = np.asarray(mids, dtype=float)
    spreads = np.broadcast_to(np.asarray(spread, dtype=float), mids.shape)
    return pd.DataFrame({
        "time": times,
        "mid": mids,
        "bid": mids - spreads / 2,
        "ask": mids + spreads / 2,
    })

def test_pricing_identities():
    args = dict(S=100.0, K=100.0, tau=3/365, sigma=0.12, sample_split="synthetic", synthetic=True)
    p = garman_kohlhagen(**args)
    assert all(np.isfinite(list(p.values())))
    assert p["call_value"] >= 0 and p["put_value"] >= 0
    assert abs(p["straddle_value"] - p["call_value"] - p["put_value"]) < 1e-12
    assert abs(p["call_value"] - p["put_value"] - (args["S"] - args["K"])) < 1e-12
    eps = 1e-4
    up = garman_kohlhagen(**{**args, "S": args["S"] + eps})["straddle_value"]
    dn = garman_kohlhagen(**{**args, "S": args["S"] - eps})["straddle_value"]
    assert abs((up - dn) / (2 * eps) - p["straddle_delta"]) < 1e-7
record_test("pricing_identities_and_finite_difference_delta", test_pricing_identities)

flat_zero = synthetic_path(
    ["2025-01-06 15:00Z", "2025-01-06 20:00Z", "2025-01-07 15:00Z"],
    [100, 100, 100], spread=0.0
)
flat_long, flat_long_ledger = run_delta_hedge(
    flat_zero, K=100, sigma0=0.12, position="long",
    sample_split="synthetic", synthetic=True
)
def test_flat_zero_spread():
    assert abs(flat_long["spot_hedge_cashflow_mid"]) < 1e-12
    assert abs(flat_long["pnl_mid"] - (flat_long["terminal_payoff"] - flat_long["premium"])) < 1e-12
record_test("flat_spot_zero_spread_cashflows", test_flat_zero_spread)

flat_spread = synthetic_path(flat_zero["time"], [100, 100, 100], spread=0.02)
flat_cost, _ = run_delta_hedge(
    flat_spread, K=100, sigma0=0.12, position="long",
    sample_split="synthetic", synthetic=True
)
def test_positive_spread_cost():
    assert flat_cost["spot_hedge_spread_cost"] > 0
    assert abs(flat_cost["pnl_bidask"] - (flat_cost["pnl_mid"] - flat_cost["spot_hedge_spread_cost"])) < RECON_TOL
record_test("positive_spread_reconciliation", test_positive_spread_cost)

moving = synthetic_path(
    ["2025-01-06 15:00Z", "2025-01-06 18:00Z", "2025-01-07 09:00Z", "2025-01-07 15:00Z"],
    [100.0, 101.0, 99.5, 100.5], spread=0.02
)
moving_long, moving_long_ledger = run_delta_hedge(
    moving, K=100, sigma0=0.15, position="long",
    sample_split="synthetic", synthetic=True
)
moving_short, moving_short_ledger = run_delta_hedge(
    moving, K=100, sigma0=0.15, position="short",
    sample_split="synthetic", synthetic=True
)
def test_long_short_identities():
    assert abs(moving_short["pnl_mid"] + moving_long["pnl_mid"]) < RECON_TOL
    assert np.allclose(moving_long_ledger["trade_q"].abs(), moving_short_ledger["trade_q"].abs(), atol=RECON_TOL)
    C = moving_long["spot_hedge_spread_cost"]
    assert abs(C - moving_short["spot_hedge_spread_cost"]) < RECON_TOL
    assert abs(moving_long["pnl_bidask"] + moving_short["pnl_bidask"] + 2 * C) < RECON_TOL
    assert not np.isclose(moving_short["pnl_bidask"], -moving_long["pnl_bidask"], atol=RECON_TOL)
record_test("long_short_midpoint_and_cost_identities", test_long_short_identities)

def test_boundary_transactions_and_no_expiry_rebalance():
    assert moving_long_ledger["transaction_type"].tolist() == [
        "initial_hedge", "intermediate_rebalance", "intermediate_rebalance", "terminal_unwind"
    ]
    assert moving_long_ledger.iloc[-1]["time"] == moving["time"].iloc[-1]
    assert pd.isna(moving_long_ledger.iloc[-1]["delta_target"])
    assert abs(moving_long_ledger.iloc[-1]["holding_after"]) < RECON_TOL
record_test("boundary_transactions_and_terminal_unwind_only", test_boundary_transactions_and_no_expiry_rebalance)

def test_calendar_time_decay():
    remaining = moving_long_ledger["remaining_tau"].to_numpy()
    assert np.all(np.diff(remaining) < 0)
    expected = (moving["time"].iloc[-1] - moving["time"].iloc[0]).total_seconds() / (365 * 24 * 3600)
    assert abs(remaining[0] - expected) < 1e-15
record_test("actual_calendar_time_decay", test_calendar_time_decay)

weekend = synthetic_path(
    ["2025-01-10 15:00Z", "2025-01-10 21:00Z", "2025-01-13 14:00Z", "2025-01-13 15:00Z"],
    [100.0, 100.2, 99.9, 100.1], spread=0.01
)
weekend_summary, weekend_ledger = run_delta_hedge(
    weekend, K=100, sigma0=0.13, position="long",
    sample_split="synthetic", synthetic=True
)
def test_weekend_path():
    assert weekend_summary["maturity_hours"] == 72
    assert weekend_ledger["time"].tolist() == weekend["time"].tolist()
    assert (weekend_ledger["time"].iloc[2] - weekend_ledger["time"].iloc[1]).total_seconds() / 3600 == 65
    assert len(weekend_ledger) == len(weekend)
record_test("weekend_carry_without_interpolation", test_weekend_path)

def test_invalid_pricing_inputs():
    base = dict(S=100, K=100, tau=1/365, sigma=0.1, sample_split="synthetic", synthetic=True)
    for field, bad in [("S", 0), ("K", 0), ("tau", 0), ("sigma", 0)]:
        try:
            garman_kohlhagen(**{**base, field: bad})
        except ValueError:
            pass
        else:
            raise AssertionError(f"Invalid {field} did not raise ValueError.")
record_test("invalid_pricing_inputs", test_invalid_pricing_inputs)

def test_invalid_spreads():
    bad = synthetic_path(
        ["2025-01-06 15:00Z", "2025-01-07 15:00Z"], [100, 100], spread=0.01
    )
    bad.loc[0, ["bid", "ask"]] = [100.01, 99.99]
    try:
        run_delta_hedge(
            bad, K=100, sigma0=0.1, position="long",
            sample_split="synthetic", synthetic=True
        )
    except ValueError as exc:
        assert "ask" in str(exc)
    else:
        raise AssertionError("Negative spread / ask < bid did not raise ValueError.")
record_test("negative_spread_and_crossed_quote", test_invalid_spreads)

unit_results.append({"test": "TEST_pricing_economics_embargo", "status": "PASS" if pricing_embargo_guard_passed else "FAIL"})
unit_results.append({"test": "TEST_hedge_PnL_economics_embargo", "status": "PASS" if hedge_embargo_guard_passed else "FAIL"})
unit_test_summary = pd.DataFrame(unit_results)
assert unit_test_summary["status"].eq("PASS").all()
display(unit_test_summary)

,test,status
0,pricing_identities_and_finite_difference_delta,PASS
1,flat_spot_zero_spread_cashflows,PASS
2,positive_spread_reconciliation,PASS
3,long_short_midpoint_and_cost_identities,PASS
4,boundary_transactions_and_terminal_unwind_only,PASS
5,actual_calendar_time_decay,PASS
6,weekend_carry_without_interpolation,PASS
7,invalid_pricing_inputs,PASS
8,negative_spread_and_crossed_quote,PASS
9,TEST_pricing_economics_embargo,PASS


## 7. Non-TEST real-data smoke tests and Stage 1 exports

Development smoke contracts are chosen mechanically and independently of outcome: the first two chronologically eligible non-TEST entry dates plus the first eligible Friday (if distinct), after applying only the frozen contract’s availability, quote-validity, and path-gap rules. They verify timezone selection, fixed entry volatility, observed bid/ask execution, explicit cashflow reconciliation, and weekend carry. They are not a strategy or profitability comparison.

In [8]:
canonical_split = canonical.drop_duplicates("model_day").set_index(canonical["model_day"].dt.date)["sample_split"].to_dict()
iv_by_date = iv_valid.assign(date_key=iv_valid["date"].dt.date).set_index("date_key")["iv"].to_dict()
entry_quotes_valid_by_date = entry_quotes_valid.set_index("candle_end_date_ny")
expiry_quotes_valid_by_date = expiry_quotes_valid.set_index("candle_end_date_ny")
source_end_times = set(hourly["candle_end_time_utc"])

def observed_path_audit(entry_date, expiry_date):
    t0 = local_time_on(entry_date, ASSUMED_IV_TIME).tz_convert("UTC")
    t1 = local_time_on(expiry_date, ASSUMED_EXPIRY_TIME).tz_convert("UTC")
    p = hourly.loc[hourly["candle_end_time_utc"].between(t0, t1)].copy()
    transitions = p.iloc[1:]
    unexpected = int(transitions["gap_classification"].isin([
        "short unexpected weekday/other gap", "major historical discontinuity", "other interval"
    ]).sum())
    return p, unexpected

def build_observed_economic_path(entry_date, expiry_date, *, sample_split):
    assert_economics_allowed(sample_split, synthetic=False)
    p, unexpected = observed_path_audit(entry_date, expiry_date)
    if unexpected:
        raise ValueError("Unexpected non-weekend source gap in observed hedge path.")
    if len(p) < 2 or not p["quote_valid"].all():
        raise ValueError("Observed hedge path is incomplete or contains invalid quotes.")
    out = p.rename(columns={
        "candle_end_time_utc": "time", "mid_close": "mid",
        "bid_close": "bid", "ask_close": "ask"
    })[["time", "mid", "bid", "ask"]].copy()
    assert entry_date in valid_entry_quote_dates
    assert expiry_date in valid_expiry_quote_dates
    entry_quote = entry_quotes_valid_by_date.loc[entry_date]
    expiry_quote = expiry_quotes_valid_by_date.loc[expiry_date]
    assert bool(entry_quote["quote_valid"]) and bool(expiry_quote["quote_valid"])
    assert entry_quote["ask_close"] >= entry_quote["bid_close"]
    assert expiry_quote["ask_close"] >= expiry_quote["bid_close"]
    assert out["time"].iloc[0] == local_time_on(entry_date, ASSUMED_IV_TIME).tz_convert("UTC")
    assert out["time"].iloc[-1] == local_time_on(expiry_date, ASSUMED_EXPIRY_TIME).tz_convert("UTC")
    return out

development_candidates = []
for entry_date in eligible_entry_dates:
    split = canonical_split.get(entry_date)
    if str(split).lower() == "test" or split is None:
        continue
    expiry_date = first_after(expiry_observation_dates, entry_date)
    if expiry_date is None:
        continue
    p_audit, unexpected = observed_path_audit(entry_date, expiry_date)
    if unexpected == 0 and len(p_audit) >= 2 and p_audit["quote_valid"].all():
        development_candidates.append((entry_date, expiry_date, split))

assert len(development_candidates) >= 3
selected = development_candidates[:2]
first_friday = next(item for item in development_candidates if item[0].weekday() == 4)
if first_friday not in selected:
    selected.append(first_friday)
selected = selected[:3]

smoke_rows = []
smoke_ledgers = {}
for contract_number, (entry_date, expiry_date, split) in enumerate(selected, start=1):
    decision_time_ny = local_time_on(entry_date - pd.Timedelta(days=1), time(17, 0))
    entry_time_ny = local_time_on(entry_date, ASSUMED_IV_TIME)
    expiry_time_ny = local_time_on(expiry_date, ASSUMED_EXPIRY_TIME)
    assert decision_time_ny < entry_time_ny < expiry_time_ny
    assert entry_date in iv_by_date
    assert entry_date in valid_entry_quote_dates
    assert expiry_date in valid_expiry_quote_dates
    raw_atm_vol = iv_by_date[entry_date]
    sigma0 = raw_atm_vol / 100.0
    assert sigma0 > 0
    path = build_observed_economic_path(entry_date, expiry_date, sample_split=split)
    S0 = float(path["mid"].iloc[0])
    K = S0
    long_summary, long_ledger = run_delta_hedge(
        path, K=K, sigma0=sigma0, position="long", sample_split=split
    )
    short_summary, short_ledger = run_delta_hedge(
        path, K=K, sigma0=sigma0, position="short", sample_split=split
    )
    contract_id = f"DEV_{contract_number:03d}"
    smoke_ledgers[(contract_id, "long")] = long_ledger
    smoke_ledgers[(contract_id, "short")] = short_ledger

    for side, summary, ledger in [
        ("long", long_summary, long_ledger), ("short", short_summary, short_ledger)
    ]:
        reconciliation_error = abs(summary["pnl_bidask"] - (
            summary["pnl_mid"] - summary["spot_hedge_spread_cost"]
        ))
        assert reconciliation_error < RECON_TOL
        assert abs(summary["final_hedge_inventory"]) < RECON_TOL
        assert ledger["time"].iloc[0] >= path["time"].iloc[0]
        assert ledger.iloc[:-1]["time"].lt(path["time"].iloc[-1]).all()
        assert ledger.iloc[-1]["time"] == path["time"].iloc[-1]
        assert ledger.iloc[-1]["transaction_type"] == "terminal_unwind"
        assert ledger.iloc[-1]["delta_target"] is np.nan or pd.isna(ledger.iloc[-1]["delta_target"])
        assert ledger["sigma_hedge"].nunique() == 1 and abs(ledger["sigma_hedge"].iloc[0] - sigma0) < 1e-15
        assert set(ledger["time"]).issubset(source_end_times)
        smoke_rows.append({
            "contract_id": contract_id,
            "selection_rule": "first_two_eligible_plus_first_eligible_friday",
            "sample_split": split,
            "position": side,
            "decision_time_ny": decision_time_ny,
            "entry_date": entry_date,
            "entry_time_ny": entry_time_ny,
            "expiry_date": expiry_date,
            "expiry_time_ny": expiry_time_ny,
            "entry_iv_date": entry_date,
            "maturity_hours": summary["maturity_hours"],
            "n_ledger_transactions": summary["n_ledger_transactions"],
            "has_expected_weekend_gap": bool(
                observed_path_audit(entry_date, expiry_date)[0].iloc[1:]["gap_classification"]
                .eq("expected Friday-to-Sunday/Monday closure").any()
            ),
            "pnl_mid_JPY_per_USD_notional": summary["pnl_mid"],
            "pnl_after_spot_cost_JPY_per_USD_notional": summary["pnl_bidask"],
            "spot_hedge_spread_cost_JPY_per_USD_notional": summary["spot_hedge_spread_cost"],
            "premium_normalised_pnl_mid": summary["premium_normalised_pnl_mid"],
            "premium_normalised_pnl_after_spot_cost": summary["premium_normalised_pnl_bidask"],
            "reconciliation_abs_error": reconciliation_error,
            "final_hedge_inventory": summary["final_hedge_inventory"],
        })

development_smoke = pd.DataFrame(smoke_rows)
assert development_smoke["reconciliation_abs_error"].max() < RECON_TOL
assert development_smoke["final_hedge_inventory"].abs().max() < RECON_TOL
assert development_smoke["entry_date"].eq(development_smoke["entry_iv_date"]).all()
assert development_smoke["sample_split"].str.lower().ne("test").all()
print(f"PASS — {development_smoke['contract_id'].nunique()} deterministic non-TEST smoke contracts reconciled within {RECON_TOL:g}.")

PASS — 3 deterministic non-TEST smoke contracts reconciled within 1e-10.


In [9]:
OBSERVED_ASSUMED_OUT = PROCESSED / "22_stage1_observed_vs_assumed.csv"
TEST_FEASIBILITY_OUT = PROCESSED / "22_stage1_test_feasibility_mapping.csv"
UNIT_TEST_OUT = PROCESSED / "22_stage1_unit_test_summary.csv"
INVENTORY_OUT = PROCESSED / "22_stage1_data_inventory.csv"
METADATA_OUT = PROCESSED / "22_stage1_run_metadata.csv"
SMOKE_OUT = PROCESSED / "22_stage1_development_engine_smoke_tests.csv"

observed_vs_assumed.to_csv(OBSERVED_ASSUMED_OUT, index=False)
test_feasibility.to_csv(TEST_FEASIBILITY_OUT, index=False)
unit_test_summary.to_csv(UNIT_TEST_OUT, index=False)
inventory.to_csv(INVENTORY_OUT, index=False)
development_smoke.to_csv(SMOKE_OUT, index=False)

run_metadata = pd.DataFrame([
    ("notebook", "22_option_contract_and_pnl_engine.ipynb"),
    ("stage", "1"),
    ("run_id_utc", RUN_ID),
    ("test_economics_unlocked", str(TEST_ECONOMICS_UNLOCKED)),
    ("assumed_iv_time", f"{ASSUMED_IV_TIME.strftime('%H:%M')} {NY_TZ}"),
    ("assumed_expiry_time", f"{ASSUMED_EXPIRY_TIME.strftime('%H:%M')} {NY_TZ}"),
    ("iv_time_status", "ASSUMPTION"),
    ("expiry_time_status", "ASSUMPTION"),
    ("atm_method", ATM_METHOD),
    ("rate_method", RATE_METHOD),
    ("day_count_method", DAY_COUNT_METHOD),
    ("hedge_frequency", HEDGE_FREQUENCY),
    ("hedge_vol_method", HEDGE_VOL_METHOD),
    ("spot_cost_method", SPOT_COST_METHOD),
    ("option_cost_method", OPTION_COST_METHOD),
    ("allow_stale_iv", str(ALLOW_STALE_IV)),
    ("allow_spot_interpolation", str(ALLOW_SPOT_INTERPOLATION)),
    ("allow_iv_interpolation", str(ALLOW_IV_INTERPOLATION)),
    ("test_pnl_exported", "False"),
    ("test_option_premium_exported", "False"),
    ("n_frozen_test_episodes", "17"),
    ("n_test_evaluable_expected", "16"),
    ("accounting_unit", ACCOUNTING_UNIT),
], columns=["field", "value"])
run_metadata.to_csv(METADATA_OUT, index=False)

stage1_outputs = [
    OBSERVED_ASSUMED_OUT, TEST_FEASIBILITY_OUT, UNIT_TEST_OUT,
    INVENTORY_OUT, METADATA_OUT, SMOKE_OUT
]
assert all(p.exists() for p in stage1_outputs)
print("Stage 1 CSV exports written.")

Stage 1 CSV exports written.


## 8. Stage 1 closure assertions

In [10]:
# Frozen N21 state and episode-start chronology.
assert len(candidate_daily) == 18 and len(episodes) == 17
assert episodes["candidate_type"].value_counts().to_dict() == {"IMPLIED_ONLY": 13, "REALISED_ONLY": 4}
assert pd.Series(reconstructed_starts).reset_index(drop=True).equals(episodes["start_day"].reset_index(drop=True))
for row in test_feasibility.itertuples():
    assert row.signal_time_ny == local_time_on(row.signal_model_day, time(17, 0))

# Clean audited ATM provenance and valid exact-10:00 quote contract.
assert iv_valid["date"].min() == pd.Timestamp("1998-12-14")
assert iv_valid["date"].max() == pd.Timestamp("2026-07-01")
assert not iv_valid["date"].dt.year.eq(1970).any()
assert iv_valid["date"].duplicated().sum() == 0 and iv_valid["iv"].gt(0).all()
assert entry_quotes_valid["quote_valid"].all()
assert (entry_quotes_valid["ask_close"] >= entry_quotes_valid["bid_close"]).all()
assert valid_entry_quote_dates.issubset(all_entry_quote_dates)
assert expiry_quotes_valid["quote_valid"].all()
assert (expiry_quotes_valid["ask_close"] >= expiry_quotes_valid["bid_close"]).all()
assert valid_expiry_quote_dates.issubset(all_expiry_quote_dates)
if ASSUMED_IV_TIME == ASSUMED_EXPIRY_TIME:
    assert valid_entry_quote_dates == valid_expiry_quote_dates

# First legitimate entry and next valid exact 10:00 expiry; same-day entry is forbidden.
for row in test_feasibility.loc[test_feasibility["evaluable"]].itertuples():
    assert row.entry_date == first_after(eligible_entry_dates, row.signal_model_day)
    assert row.entry_date > row.signal_model_day
    assert row.option_entry_time_ny == local_time_on(row.entry_date, ASSUMED_IV_TIME)
    assert row.expiry_date == first_after(expiry_observation_dates, row.entry_date)
    assert row.expiry_time_ny == local_time_on(row.expiry_date, ASSUMED_EXPIRY_TIME)
    assert row.signal_time_ny < row.option_entry_time_ny < row.expiry_time_ny
    assert row.entry_iv_available and row.entry_spot_available and row.expiry_spot_available
    assert row.entry_date in valid_entry_quote_dates
    assert row.expiry_date in valid_expiry_quote_dates
    assert row.unexpected_gap_count == 0 and row.hedge_path_available

# Coverage and the non-rescue of the final episode.
assert len(test_feasibility) == 17
assert int(test_feasibility["evaluable"].sum()) == 16
e17 = test_feasibility.loc[test_feasibility["episode_id"].eq("E21_017")].iloc[0]
assert not e17["evaluable"]
assert e17["unevaluable_reason"] == "No later date has both same-date raw O/N ATM IV and exact observed 10:00 NY spot close."

# Embargo is enforced in computation and export schema.
assert TEST_ECONOMICS_UNLOCKED is False
assert pricing_embargo_guard_passed and hedge_embargo_guard_passed
exported_test = pd.read_csv(TEST_FEASIBILITY_OUT)
assert not any(fragment in c.lower() for c in exported_test.columns for fragment in BANNED_TEST_FRAGMENTS)
assert exported_test.columns.tolist() == test_feasibility.columns.tolist()
assert not any("test" in p.name.lower() and any(x in p.name.lower() for x in ["pnl", "premium", "strategy"])
               for p in PROCESSED.glob("22_stage1*"))

# Pricing, hedge accounting, fixed volatility, inventory and no interpolation.
assert unit_test_summary["status"].eq("PASS").all()
assert development_smoke["reconciliation_abs_error"].max() < RECON_TOL
assert development_smoke["final_hedge_inventory"].abs().max() < RECON_TOL
for ledger in smoke_ledgers.values():
    assert ledger["sigma_hedge"].nunique() == 1
    assert set(ledger["time"]).issubset(source_end_times)
    assert ledger.iloc[-1]["transaction_type"] == "terminal_unwind"
    assert pd.isna(ledger.iloc[-1]["delta_target"])
    assert abs(ledger.iloc[-1]["holding_after"]) < RECON_TOL

print(f"""Stage 1 contract frozen.

- IV observation time: assumed {ASSUMED_IV_TIME.strftime('%H:%M')} {NY_TZ}
- synthetic expiry cut: assumed next tradable {ASSUMED_EXPIRY_TIME.strftime('%H:%M')} {NY_TZ}
- ATM convention: K = entry spot
- rates: zero domestic/foreign baseline
- option volatility: raw Bloomberg O/N ATM IV
- maturity: ACT/365 actual assumed elapsed calendar time
- hedge: observed hourly closes, entry IV fixed
- spot hedge costs: observed bid/ask
- option execution costs: unavailable / excluded
- stale IV: prohibited
- interpolation: prohibited
- TEST economics: LOCKED

Frozen TEST hidden-event feasibility:
17 episodes
16 evaluable
1 unevaluable

No TEST option premium or PnL has been calculated.""")

Stage 1 contract frozen.

- IV observation time: assumed 10:00 America/New_York
- synthetic expiry cut: assumed next tradable 10:00 America/New_York
- ATM convention: K = entry spot
- rates: zero domestic/foreign baseline
- option volatility: raw Bloomberg O/N ATM IV
- maturity: ACT/365 actual assumed elapsed calendar time
- hedge: observed hourly closes, entry IV fixed
- spot hedge costs: observed bid/ask
- option execution costs: unavailable / excluded
- stale IV: prohibited
- interpolation: prohibited
- TEST economics: LOCKED

Frozen TEST hidden-event feasibility:
17 episodes
16 evaluable
1 unevaluable

No TEST option premium or PnL has been calculated.
